## Successful Semantic Modelling for Power BI - "00. Setup"

THE ONLY NOTEBOOK AN ATTENDEE COPIES BY HAND. Everything else it fetches.

HOW IT IS USED ON THE DAY
1. Attendee signs in with their numbered login (userNNNN).
2. Attendee creates their OWN new, empty workspace.
3. From the shared workspace (they have view access), they copy THIS notebook
into their own workspace.
4. Run all. It imports the lab notebooks from GitHub, files them into a
"Labs" folder, and shrinks the workspace Spark defaults so the whole room
fits on the shared capacity.
5. They then open "0-create-lab-models" and work through the labs.

Run this as a **Python** notebook, not PySpark: it is all REST and library
calls, so it starts in seconds and needs no Spark session.

In [ ]:
# ---- CELL 1: INSTALL --------------------------------------------------------
# Alone, and first. In Fabric %pip restarts the Python interpreter, so anything
# defined before it is lost. Config and imports therefore live in Cell 2.
%pip install -q semantic-link-labs

In [ ]:
# ---- CELL 2: CONFIG + IMPORTS ----------------------------------------------
# Where the lab notebooks are published. Presenter: set this to the public repo
# and confirm the raw URLs resolve BEFORE the day.
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/dax-tips/SuccessfulSemanticModelling/main"

# Display name in the workspace -> path within the repo.
LAB_NOTEBOOKS = {
    "0-create-lab-models":         "labs/notebooks/0-create-lab-models.ipynb",
    "lab02-storage-modes":         "labs/notebooks/lab02-storage-modes.ipynb",
    "lab07-diagnose-slow-visuals": "labs/notebooks/lab07-diagnose-slow-visuals.ipynb",
    "lab08-prove-the-improvement": "labs/notebooks/lab08-prove-the-improvement.ipynb",
    # Fallback only, for a tenant where OneLake shortcuts are blocked. If you
    # enable this, attendees run it INSTEAD of Cell 4 of 0-create-lab-models.
    # "bootstrap-attendee":         "labs/bootstrap/bootstrap-attendee.ipynb",
}

LAB_FOLDER = "Labs"          # workspace folder the imported notebooks land in
SESSION_TIMEOUT_MIN = 30     # idle Spark sessions auto-release after this
RUNTIME_VERSION = "1.3"      # Fabric Spark runtime

import sempy
import sempy_labs as labs
import sempy.fabric as fabric
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

client = fabric.FabricRestClient()
workspace_id = fabric.resolve_workspace_id()
print(f"Setting up workspace {workspace_id}")

In [ ]:
# ---- CELL 3: IMPORT THE LAB NOTEBOOKS FROM GITHUB ---------------------------
for name, path in LAB_NOTEBOOKS.items():
    labs.import_notebook_from_web(
        notebook_name=name, url=f"{GITHUB_RAW_BASE}/{path}", overwrite=True)
    print(f"  imported: {name}")

print(f"{len(LAB_NOTEBOOKS)} lab notebook(s) imported.")

In [ ]:
# ---- CELL 4: FILE THEM INTO A "LABS" FOLDER ---------------------------------
resp = client.post(f"/v1/workspaces/{workspace_id}/folders",
                   json={"displayName": LAB_FOLDER})
if resp.status_code in (200, 201):
    folder_id = resp.json()["id"]
    print(f"Created '{LAB_FOLDER}' folder")
else:
    folders = client.get(f"/v1/workspaces/{workspace_id}/folders").json()
    folder_id = next(f["id"] for f in folders["value"] if f["displayName"] == LAB_FOLDER)
    print(f"Using existing '{LAB_FOLDER}' folder")

items = client.get(f"/v1/workspaces/{workspace_id}/items?type=Notebook").json()
for item in items["value"]:
    if item["displayName"] in LAB_NOTEBOOKS:
        r = client.post(f"/v1/workspaces/{workspace_id}/items/{item['id']}/move",
                        json={"targetFolderId": folder_id})
        print(f"  {'moved' if r.status_code == 200 else f'error {r.status_code}'}: {item['displayName']}")

In [ ]:
# ---- CELL 5: SHRINK THIS WORKSPACE'S SPARK FOOTPRINT ------------------------
# One node, one executor, short timeout. Thirty of these have to coexist on the
# shared capacity, so a default-sized pool per attendee would swamp it.
body = {
    "automaticLog": {"enabled": False},
    "highConcurrency": {"notebookInteractiveRunEnabled": True,
                        "notebookPipelineRunEnabled": True},
    "pool": {"customizeComputeEnabled": False,
             "defaultPool": {"name": "Starter Pool", "type": "Workspace"},
             "starterPool": {"maxNodeCount": 1, "maxExecutors": 1}},
    "environment": {"runtimeVersion": RUNTIME_VERSION},
    "job": {"conservativeJobAdmissionEnabled": False,
            "sessionTimeoutInMinutes": SESSION_TIMEOUT_MIN},
}
print(client.patch(f"/v1/workspaces/{workspace_id}/spark/settings", json=body))

# Large storage format is what lets a Direct Lake model page columns on demand.
labs.set_workspace_default_storage_format(storage_format="Large")
print("Spark defaults shrunk; storage format set to Large.")

In [ ]:
# ---- CELL 6: READY CHECK ----------------------------------------------------
settings = client.get(f"/v1/workspaces/{workspace_id}/spark/settings").json()
pool = settings.get("pool", {}).get("starterPool", {})
print("Setup complete")
print("-" * 46)
print(f"  starter maxNodes : {pool.get('maxNodeCount')}")
print(f"  session timeout  : {settings.get('job', {}).get('sessionTimeoutInMinutes')} min")
print(f"  notebooks in '{LAB_FOLDER}': {', '.join(LAB_NOTEBOOKS)}")
print("-" * 46)
print("Next: open '0-create-lab-models' and Run all. It takes 10-15 minutes,")
print("so start it now and let it run while the session gets going.")